# Bronze Ingestion — Netflix Titles (Job version)

Production ingestion notebook — loads any new files from the landing folder into the Bronze Delta table.

## 1. Configuration

Defines all paths and table names in one place

In [0]:
storage_account = "dlspl21databricks"
container = "ivanrazumovskyi"
catalog = "dbr_dev"
bronze_schema = "ivanrazumovskyi_bronze"
target_table = "netflix_titles"

storage_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

source_path = f"{storage_root}/ingestion/netflix/"
checkpoint_path = f"{storage_root}/checkpoints/netflix_titles/"
bronze_table = f"{catalog}.{bronze_schema}.{target_table}"

print("Source path:", source_path)
print("Checkpoint path:", checkpoint_path)
print("Target table:", bronze_table)

## 2. Define the explicit source schema

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

source_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True),
])

## 3. Read new files with Auto Loader and add metadata columns

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

df_bronze = (
    df_bronze
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

## 4. Write the stream to the Bronze Delta table

`availableNow=True` processes whatever new files exist since the last run, then stops — safe to run manually or on a schedule.

In [0]:
query = (
    df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
-- Row count check
SELECT COUNT(*) AS row_count FROM dbr_dev.ivanrazumovskyi_bronze.netflix_titles;